# 79 - Segmentación con Mask R-CNN (PyTorch) — Base

Uso de **Mask R-CNN** preentrenado en COCO con PyTorch. Ejemplo de inferencia sobre una imagen pública.

In [ ]:

try:
    import torch, torchvision
    from torchvision import transforms
    from torchvision.utils import draw_segmentation_masks
    from PIL import Image
    import requests, io
    HAS_TORCH = True
    print("✅ PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
except Exception as e:
    HAS_TORCH = False
    print("⚠️ PyTorch/torchvision no disponible:", e)

if HAS_TORCH:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=weights).to(device).eval()
        preprocess = weights.transforms()
    except Exception as e:
        model = None; preprocess=None
        print("⚠️ No se pudo cargar Mask R-CNN:", e)

    url = "https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg"
    try:
        img = Image.open(io.BytesIO(requests.get(url,timeout=10).content)).convert("RGB")
    except Exception as e:
        img = None; print("⚠️ No se pudo descargar imagen:", e)

    if model and img:
        t = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(t)[0]
        keep = out["scores"] > 0.7
        masks = (out["masks"][keep]>0.5).squeeze(1).cpu()
        img_t = transforms.ToTensor()(img)
        if len(masks)>0:
            annotated = draw_segmentation_masks((img_t*255).byte(), masks, alpha=0.5)
            from torchvision.utils import save_image
            save_image(annotated.float()/255.0, "/mnt/data/maskrcnn_output.png")
            print("✅ Segmentación realizada. Guardado en /mnt/data/maskrcnn_output.png")
else:
    print("Ejecuta en Colab/local con PyTorch y torchvision instalados.")


## 📝 Ejercicios
1) Cambia el umbral de `score` a 0.3 y observa.
2) Prueba imágenes propias.
3) Genera un mapa de calor de las máscaras.